# SANCOCHO PRELIMINARY DATA IMPORT

# 0.0 Initial Library Import

In [ ]:
# NOTE FOR CODEX / DEVELOPERS
# ---------------------------
# This notebook is derived from the sancocho reference pipeline.
# The objective is to adapt the same pipeline structure to train
# a market regime model.
#
# The core pipeline steps should remain consistent:
#   1. data import
#   2. cleanup
#   3. feature engineering
#   4. target creation
#   5. model training
#   6. model optimization
#   7. artifact export
#
# Reuse functions from notebook_utils wherever possible.

In [ ]:
# Standard libraries
import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_UTILS_SRC = PROJECT_ROOT / "notebook_utils" / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))
if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))
from paths import DATA_RAW, DATA_INTERMEDIATE, FIGURES, MODELS, OUTPUTS, TABLES

import numpy as np
import pandas as pd
import operator
from datetime import datetime, date
from IPython.display import display, HTML
import pyfolio as pf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, norm, entropy
from scipy.optimize import minimize
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

warnings.filterwarnings('ignore')

# 3.7.16 specific
from typing import List

In [ ]:
# Panda display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters

print("sklearn:", sklearn.__version__)
print("xgb:", xgboost.__version__)
print("pandas:", pd.__version__)
print("numpy :", np.__version__)

# 0.1 Variable selection

## - Function import: formatting

In [ ]:
# My Functions: same directory
user = 'mateo'
model = "regime_model"
date_file = '20260213'
model_round = 'round_1'
# ticker = 'TSLA'
# import _formatting_functions
# importlib.reload(_formatting_functions)
# from _formatting_functions import color_negative_red, make_pretty, data_format, quarter_hr, event_import

model_input_path = DATA_RAW / "xls" / "input" / model
model_varnames_path = model_input_path / "_varnames_"
model_round_input_path = model_input_path / model_round
model_intermediate_path = DATA_INTERMEDIATE / model / model_round
model_table_output_path = TABLES / model / model_round  # REVIEW: confirm local target path for prior xls/output exports
model_csv_output_path = model_table_output_path / "csv"
model_xls_output_path = model_table_output_path / "xls"
model_figure_output_path = FIGURES / model / model_round
model_ml_output_path = MODELS / model / model_round
research_liquidity_path = DATA_RAW / "research" / "liquidity" / "xls"
research_fear_greed_path = DATA_RAW / "research" / "fear_greed" / "csv"

for path_obj in [
    model_round_input_path,
    model_intermediate_path,
    model_table_output_path,
    model_csv_output_path,
    model_xls_output_path,
    model_figure_output_path,
    model_ml_output_path,
]:
    path_obj.mkdir(parents=True, exist_ok=True)

import importlib
import notebook_utils._formatting_functions as fmt

importlib.reload(fmt)

from notebook_utils._formatting_functions import (color_negative_red, make_pretty, data_format, quarter_hr, event_import)


In [ ]:
# Importing master list of columns/features

vars_path = model_varnames_path
vars_name = f'master_variable_inventory {date_file}.xlsx' #Master data dictionary to format columns of backtest

master_col_df = pd.read_excel(vars_path / vars_name, sheet_name='varnames', usecols=['kite name', 'clean name', 'category', 'code', 'distance'])

In [ ]:
# Importing raw data

vars_path = model_round_input_path
vars_name = 'Sancocho_v1_LONG_2013_2026_expanded_universe.csv' #Master data dictionary to format columns of backtest

backtest_df = pd.read_csv(vars_path / vars_name)

In [ ]:
# Correcting variable names

backtest_df.drop(columns=["entry_time"], inplace=True)
backtest_df.drop(columns=["exit_time"], inplace=True)
backtest_df['entry_pl'] = backtest_df['mtm_pl']
backtest_df.rename(columns={"entry_date_time": "entry_time"}, inplace=True)
backtest_df.rename(columns={"exit_date_time": "exit_time"}, inplace=True)

In [ ]:
# Exporting modified raw data
vars_path = model_intermediate_path
outname = f"Sancocho_v1_LONG_2013_2026_expanded MODIFIED {datetime.now().strftime('%Y%m%d')}.csv"
backtest_df.to_csv(vars_path / outname, index=False)

In [ ]:
# Merging master variable list classification with backtest data

column_names = backtest_df.columns.tolist()
variable_inventory_df = pd.DataFrame(column_names, columns=["kite name"])
print(len(variable_inventory_df))

variable_inventory_df = variable_inventory_df.merge(master_col_df, on="kite name", how="left")
print(len(variable_inventory_df))

# 119

## - Clean data dictionary export

In [ ]:
# Exporting variable inventory - produces clean variable dictionary for import

vars_path = DATA_INTERMEDIATE / model
vars_path.mkdir(parents=True, exist_ok=True)
outname = f"variable_inventory_expanded_{datetime.now().strftime('%Y%m%d')}.xlsx"

variable_inventory_df.to_excel(vars_path / outname, index=False, engine='openpyxl')

# 1.0 Importing Data From Backtest [START HERE IF RERUNNING!]

In [ ]:
# Data columns by groups: front(most important), kite (always the same), sym_ta_cols (ta colums for symbol), other symbol ta (extra_sym_cols), add-ons (extra), etc..
## Excel file contains the categories
vars_path = DATA_INTERMEDIATE / model

## This file is created in step 0.1
vars_name = 'variable_inventory_expanded_20260213.xlsx'

df = pd.read_excel(vars_path / vars_name, sheet_name='Sheet1', usecols=['kite name', 'code'])
df.drop(df[df['kite name'] == 'MISSING'].index, inplace=True)
print(len(df))

# 8 Lists of columns are created: front_cols, kite_cols, sym_ta_cols, extra_sym_cols, sec_cols, extra_cols, drop_cols, last_cols
front_cols = None
kite_cols = None
sym_ta_cols = None
extra_sym_cols = None
extra_cols = None
sec_cols = None
last_cols = None
drop_cols = None

grouped = df.groupby('code')['kite name'].apply(list)
list_dict = grouped.to_dict()
for key, value in list_dict.items():
    globals()[key] = value

# Important step: Ensures 'symbol' and 'mtm_pl' are first on the list... which will ultimately be 'normed_date' and 'symbol' are the top columns
front_cols.sort(reverse=True)
#['symbol', 'mtm_pl', 'matched_shares', 'entry_pl', 'entry_collect.Daily_open']
print(grouped)


In [ ]:
# Remember: Importing MODIFIED data

model_round = 'round_1'
path = model_intermediate_path

filenames = [
            'Sancocho_v1_LONG_2013_2026_expanded MODIFIED 20260213.csv',
             ]

event_data = pd.DataFrame()

for file in filenames:
    file_path = path / file
    new_data = pd.read_csv(file_path)
    new_data['entry_time'] = pd.to_datetime(new_data['entry_time'])
    new_data['exit_time'] = pd.to_datetime(new_data['exit_time'])
    new_data['normed_date'] = new_data['entry_time'].dt.date
    new_data.sort_index(inplace=True)
    # extra_cols = None
    # all_columns = ['normed_date'] + front_cols + kite_cols + sym_ta_cols + extra_sym_cols + extra_cols + sec_cols + drop_cols
    all_columns = ['normed_date'] + front_cols + kite_cols + sym_ta_cols + extra_sym_cols + sec_cols + drop_cols
    new_data = new_data[all_columns]
    new_data = new_data.drop(columns=drop_cols)
    new_data.columns = new_data.columns.str.replace('entry_collect.', '')
    event_data = pd.concat([event_data, new_data], ignore_index = False)

event_data.sort_values(by=['entry_time', 'symbol'], ascending=[False, True], inplace=True) #Sorts descending by vars

In [ ]:
# 111 initial variables imported / 888698 observations

# Creating a placeholder at the end of the data - dashboard function drops last column of every df - this is a legacy from the previous runs that had this data available (data will be added in future runs)
event_data['modelspec'] = None

columns_list = event_data.columns.tolist()
print(f'variables: {len(columns_list)}')
print(f'observations: {event_data.shape[0]}')
print(textwrap.fill(", ".join(columns_list), width = 250))


In [ ]:
print(event_data["normed_date"].head(2))
print(event_data["normed_date"].tail(2))

### - Exporting event data

In [ ]:
# Exported cleaned data to verify column aggregation (sum vs max to keep one row)
csv_outpath = model_csv_output_path
dataname = f'sancocho_long '
outname = f"- top expanded event_data {datetime.now().strftime('%Y%m%d')}.csv"
# event_data.to_csv(csv_outpath / f"{dataname}{outname}", index=False)
# event_data.head(1000).to_csv(csv_outpath / f"{dataname}{outname}", index=False)

outname = f"- bottom expanded event_data {datetime.now().strftime('%Y%m%d')}.csv"
# event_data.tail(1000).to_csv(csv_outpath / f"{dataname}{outname}", index=False)

# 2.0 Analysing Raw Data

### - Function import: dashboard (multi-symbol v1)

In [ ]:
## My Functions: same Phalanx directory
# For dashboards
# import _dashboard_functions_v1
# importlib.reload(_dashboard_functions_v1)
# from _dashboard_functions_v1 import dashboard

# # For deciles, charts and distances
# import _data_explore_functions
# importlib.reload(_data_explore_functions)
# from _data_explore_functions import cross_tabs, explore_cross, decile_summary, line_chart_grid, create_distance, append_summary, get_summary, reset_summary, count_outliers_by_std

import notebook_utils._dashboard_functions_v1 as dash
import notebook_utils._data_explore_functions as explore

importlib.reload(dash)
importlib.reload(explore)

from notebook_utils._dashboard_functions_v1 import dashboard
from notebook_utils._data_explore_functions import (cross_tabs, explore_cross, decile_summary, line_chart_grid, create_distance, append_summary, get_summary, reset_summary, count_outliers_by_std,)

### - Modifying certain columns to match functions

In [ ]:
# NOTE FOR CODEX / DEVELOPERS: This is a short model, so entry side is -1 (ie. selling); long_short variable should be -1 across
# Data represents long set up
event_data['entry_side'] = -1
event_data['exit_side'] = 1


In [ ]:
## Creating Dashboard
# Unoptimized data

# Runding dashboard function
strat_000, pnl_symboldate_000, pnl_bydate_000 = dashboard(event_data, comm=3.5, bpower=1000000, mod_id='no_optmz', long_short=-1, dash_type='long')
# strat_000, pnl_symboldate_000, pnl_bydate_000 = dashboard(event_data, 3.5, 'no_optmz', 1, 'complete', 'max')

print(f'full data: {len(event_data)}')
print(f'new data: {len(pnl_symboldate_000)}')

# NOTE: Gross P/L calc matches original
# full data: 893306
# new data: 892373

### - Data export

In [ ]:

# General parameters/locations
xls_outpath = model_xls_output_path
csv_outpath = model_csv_output_path

outname = f"- pnl_symboldate_000 {datetime.now().strftime('%Y%m%d')}.csv"
# pnl_symboldate_000.to_csv(csv_outpath / f"{dataname}{outname}", index=False)

outname = f"- pnl_date_000 {datetime.now().strftime('%Y%m%d')}.xlsx"
# pnl_bydate_000.to_excel(xls_outpath / f"{dataname}{outname}", index=False, engine='openpyxl')


### - Data copy 1

In [ ]:
# Data copy 1
static_rvw_001 = pnl_symboldate_000.copy()


### - Variable imputation: Ask, SPX_prev_close and SI_pct have missing values

In [ ]:
# Picking best available option for imputation - nothing is perfect

# entry_price could be used, opted for earlier value
# static_rvw_001["Ask"] = static_rvw_001["Ask"].fillna(static_rvw_001["Last_px"])

# assumes the SPX did not move in a day
# static_rvw_001["SPX_prev_close"] = static_rvw_001["SPX_prev_close"].fillna(static_rvw_001["SPY_close"])

# assumes no short interest 
# static_rvw_001["SI_pct"] = static_rvw_001["SI_pct"].fillna(0)

summary_df = append_summary(event_data, "event_data")
summary_df = append_summary(static_rvw_001, "static_rvw_001")
print(summary_df)

#              Name     obs ncols
# 0      event_data  893306   117
# 1  static_rvw_001  892373   127

# 2.1 Distance Variables: Symbol related

In [ ]:
# Creating Data Frame (df) with list of variables from imported excel file
df = pd.read_excel(vars_path / vars_name, sheet_name='Sheet1', usecols=['clean name', 'category', 'distance', 'kite name'])
df.drop(df[df['kite name'] == 'MISSING'].index, inplace=True)

# Symbol distance variables (28)
df_symbol_dist = df[(df['distance'] == True) & (df['category'] == 'symbol')][['clean name']]
symbol_dist_cols = df_symbol_dist['clean name'].tolist()
print(f'symbol: {len(symbol_dist_cols)}')
print(textwrap.fill(", ".join(symbol_dist_cols), width = 250))


In [ ]:
# Uses calculated ATR (TA.Lib) to normalize, and the 'prev_close' as reference variable
for column in symbol_dist_cols:
    # to the ask
    create_distance('dist', static_rvw_001, 'Last_px', column, 'ATR');

# 34 vars
symbol_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_')]

print(len(symbol_dist_cols))
print(textwrap.fill(", ".join(symbol_dist_cols), width = 250))


# 2.2 Distance Variables: SPY related

In [ ]:
# Spy distance variables (8)

exclude = ["SPY_px", "SPY_premkt_vol"]

df_spy_dist = df[(df['distance'] == True) & (df['category'] == 'spy') & (~df['clean name'].isin(exclude))][['clean name']]
spy_dist_cols = df_spy_dist['clean name'].tolist()
print(f'spy: {len(spy_dist_cols)}')
print(textwrap.fill(", ".join(spy_dist_cols), width = 250))

for column in spy_dist_cols:
    # to the SPY level at entry
    create_distance('dist', static_rvw_001, 'SPY_px', column, 'SPY_atr');
    
spy_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_SPY')]
print(len(spy_dist_cols))
print(textwrap.fill(" , ".join(spy_dist_cols), width = 250))


# 2.3 Distance Variables: Relative (symbol vs SPY)

In [ ]:

static_rvw_001['dist_Last_px_SPY_prev_close'] = static_rvw_001['dist_Last_px_prev_close'] / static_rvw_001['dist_SPY_px_SPY_prev_close']

symbol_spy_dist_cols = [col for col in static_rvw_001.columns if col.startswith('dist_Last_px_SPY')]
print(len(symbol_spy_dist_cols))
print(textwrap.fill(", ".join(symbol_spy_dist_cols), width = 250))


# 2.4 Consolidating Distances: Symbol + SPY

In [ ]:
dist_gap = ['dist_Last_px_SPY_prev_close']
consolidated_dist = symbol_dist_cols + spy_dist_cols + symbol_spy_dist_cols

print(len(consolidated_dist))
print(textwrap.fill(", ".join(consolidated_dist), width = 250))

In [ ]:
dist_consol_summ = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in consolidated_dist:
    summary = decile_summary(static_rvw_001, var)
    dist_consol_summ[var] = summary  # Store the summary in the DataFrame with the variable name as the column

dist_consol_summ


In [ ]:
# We are deleting a variable that we just created, but only after inspecting values in previous step (bunch of NaNs and Infs)
# Removing useless distance variables
static_rvw_001.drop(columns=["dist_Last_px_Last_px", "dist_Last_px_SPY_prev_close"], inplace=True)


In [ ]:
# static_rvw_001["dist_Last_px_Arimax_pred_1"].plot.kde()


In [ ]:
outlier_table = count_outliers_by_std(static_rvw_001, consolidated_dist)
print(outlier_table)

# 2.5 Volume Variables

In [ ]:
# Collecting relevant volume variables (4): 'AVOL' left out as scaling factor
volu_vars = ['Premkt_vol', 'prev_askvol', 'prev_vol', 'Acc_vol']

for item in volu_vars:
    static_rvw_001[f'{item}_rat'] = static_rvw_001[item] / static_rvw_001['AVOL']

# 166 variables to this point / 629430 obs
list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
print(textwrap.fill(", ".join(list_vars), width = 250))


In [ ]:
# Analysis of all volume variables (7)
volu_vars = ['Premkt_vol', 'prev_askvol', 'prev_vol', 'Premkt_vol_rat', 'prev_askvol_rat', 'prev_vol_rat', 'Acc_vol_rat']

print(len(volu_vars))
print(volu_vars)

volu_consol_dist = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in volu_vars:
    summary = decile_summary(static_rvw_001, var)
    volu_consol_dist[var] = summary  # Store the summary in the DataFrame with the variable name as the column

volu_consol_dist


In [ ]:
outlier_table = count_outliers_by_std(static_rvw_001, volu_vars)
print(outlier_table)

# 2.6 Time Variables

In [ ]:
static_rvw_001['entry_hr_dec'] = static_rvw_001['entry_time'].dt.hour + static_rvw_001['entry_time'].dt.minute / 60
static_rvw_001['exit_hr_dec'] = static_rvw_001['exit_time'].dt.hour + static_rvw_001['exit_time'].dt.minute / 60
static_rvw_001['entry_hr_dec_to_close'] = 16.00 - static_rvw_001['entry_hr_dec']

static_rvw_001["year_day"]  = static_rvw_001["entry_time"].dt.dayofyear

static_rvw_001["week_day_sin"] = np.sin(2 * np.pi * static_rvw_001["week_day"] / 7)
static_rvw_001["week_day_cos"] = np.cos(2 * np.pi * static_rvw_001["week_day"] / 7)

static_rvw_001["month_sin"] = np.sin(2 * np.pi * static_rvw_001["month"] / 12)
static_rvw_001["month_cos"] = np.cos(2 * np.pi * static_rvw_001["month"] / 12)

static_rvw_001["year_day_sin"] = np.sin(2 * np.pi * static_rvw_001["year_day"] / 365.25)
static_rvw_001["year_day_cos"] = np.cos(2 * np.pi * static_rvw_001["year_day"] / 365.25)

list_vars = static_rvw_001.columns.tolist()

print(len(list_vars))
print(len(static_rvw_001))
print(textwrap.fill(", ".join(list_vars), width = 250))

# 2.7 Return Variables

### - Previous closing dates 

In [ ]:
# sample = static_rvw_001.static_rvw_001(frac=0.01, random_state=42)

In [ ]:
prev_close_vars = ['Prev_close_2', 'Prev_close_3', 'Prev_close_4', 'Prev_close_5', 'Prev_close_6', 'Prev_close_7', 'Prev_close_8', 'Prev_close_9', 'Prev_close_10']

# creates returns from prev_close to the last 9 day closing
for item in prev_close_vars:
    static_rvw_001[f'ret_{item}'] = (static_rvw_001[item] / static_rvw_001['prev_close'])-1

# creates a dummy for each increasing sequence of prices (price went up for last 2 days, 3 days... 9 days)
for i in range(2, 10):
    cols = [f"Prev_close_{j}" for j in range(2, i + 1)]
    cond = (static_rvw_001["prev_close"].values[:, None] > static_rvw_001[cols].values).all(axis=1)
    static_rvw_001[f"momo_px_{i}"] = cond.astype(int)

# 166 variables to this point / 629430 obs
list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
# print(textwrap.fill(", ".join(list_vars), width = 250))


In [ ]:
# select relevant columns
prev_close_ret_vars = sorted([c for c in static_rvw_001.columns if c.startswith("ret_Prev_close_")], key=lambda x: int(x.split("_")[-1]))

# compute positive count
static_rvw_001["pos_ret"] = (static_rvw_001[prev_close_ret_vars] > 0).sum(axis=1)

# assign weights: inverse of the lag number, so ret_Prev_close_2 > ret_Prev_close_10
weights = [1 / int(c.split("_")[-1]) for c in prev_close_ret_vars]

# normalize to sum to 1 (so result is %)
weights = [w / sum(weights) for w in weights]

# compute pos_pct
static_rvw_001["pos_pct"] = (static_rvw_001[prev_close_ret_vars].gt(0).astype(int) * weights).sum(axis=1)


### - Moving averages

In [ ]:
# Note the ORDER of the variables matter - assigns more weight to shorter EMA windows
ma_vars = ['EMA_8', 'EMA_8_2', 'EMA_20', 'EMA_20_2', 'EMA_50', 'EMA_50_2','EMA_100', 'EMA_100_2',  'EMA_200', 'EMA_200_2']

for item in ma_vars:
    static_rvw_001[f'ret_{item}'] = (static_rvw_001[item] / static_rvw_001['prev_close'])-1

for i, col in enumerate(ma_vars, start=1):
    cols = ma_vars[:i]  # take all EMAs up to this point
    cond = (static_rvw_001["prev_close"].values[:, None] > static_rvw_001[cols].values).all(axis=1)
    static_rvw_001[f"momo_ma_{i}"] = cond.astype(int)

list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
# print(textwrap.fill(", ".join(list_vars), width = 250))


In [ ]:
# select relevant columns
# ma_ret_vars = sorted([c for c in static_rvw_001.columns if c.startswith("ret_EMA_")], key=lambda x: int(x.split("_")[-1]))
ma_ret_vars = [c for c in static_rvw_001.columns if c.startswith("ret_EMA_")]

# compute positive count
static_rvw_001["pos_ma_ret"] = (static_rvw_001[ma_ret_vars] > 0).sum(axis=1)

# assign weights: inverse of the lag number, so ret_Prev_close_2 > ret_Prev_close_10
weights = [1 / int(c.split("_")[-1]) for c in ma_ret_vars]

# normalize to sum to 1 (so result is %)
weights = [w / sum(weights) for w in weights]

# compute pos_pct
static_rvw_001["pos_ma_pct"] = (static_rvw_001[ma_ret_vars].gt(0).astype(int) * weights).sum(axis=1)



### - Prices and MAs

In [ ]:
pxma_vars = ['Prev_close_3', 'Prev_close_5', 'Prev_close_9', 'EMA_20', 'EMA_50', 'EMA_200']

for i, col in enumerate(pxma_vars, start=1):
    cols = pxma_vars[:i]  # take all EMAs up to this point
    cond = (static_rvw_001["prev_close"].values[:, None] > static_rvw_001[cols].values).all(axis=1)
    static_rvw_001[f"momo_pxma_{i}"] = cond.astype(int)

list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
# print(textwrap.fill(", ".join(list_vars), width = 250))


### - TS forecasts (ARIMAX and VAR)

In [ ]:
ts_vars = ['Arimax_pred_1', 'Arimax_pred_2', 'Var_pred_1', 'Var_pred_2']

for item in ts_vars:
    static_rvw_001[f'dumm_{item}'] = (static_rvw_001[item] > static_rvw_001['prev_close']).astype(int)

list_vars = static_rvw_001.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_001)}')
# print(textwrap.fill(", ".join(list_vars), width = 250))


# 2.8 External Variables

### - Liquidity Indices

In [ ]:
# Importing master list of columns/features
vars_path = research_liquidity_path
vars_name = f'liquidity_indices_final_20260213.xlsx'
cols = ['DATE', 'PCA_Index_ma5', 'PCA_ScaledIndex_ma5', 'PCA_Index_ma20', 'PCA_ScaledIndex_ma20', 'PCA_Index_ma50', 'PCA_ScaledIndex_ma50', 'PCA_Raw_full', 'PCA_Index_full']

liquidity_df = pd.read_excel(vars_path / vars_name, sheet_name='Sheet1', usecols=cols, parse_dates=["DATE"])

liquidity_df = liquidity_df.dropna(how="any")
liquidity_df["normed_date"] = liquidity_df["DATE"].dt.strftime("%Y-%m-%d")  # or any format


In [ ]:
# Must normalize the column types, as object masks the true type of the column, thus merge WILL FAIL
static_rvw_001["normed_date"] = pd.to_datetime(static_rvw_001["normed_date"]).dt.normalize()
liquidity_df["normed_date"] = pd.to_datetime(liquidity_df["normed_date"]).dt.normalize()

static_rvw_002 = static_rvw_001.merge(
    liquidity_df.drop(columns=["DATE"]),
    how="left",
    on="normed_date",
    suffixes=("", "_liq")
)

print(static_rvw_002[["normed_date", "PCA_Index_full"]].head(1))
print(static_rvw_002[["normed_date", "PCA_Index_full"]].tail(1))

### - Fear and Greed

In [ ]:
# Importing master list of columns/features
vars_path = research_fear_greed_path
vars_name = f'fear_and_greed_full.csv'
cols = ['date', 'value']

fear_df = pd.read_csv(vars_path / vars_name, usecols=cols)
fear_df = fear_df.dropna(how="any")
fear_df = fear_df.rename(columns={"date": "normed_date", "value": "fear_greed"})


In [ ]:
fear_df["normed_date"] = pd.to_datetime(fear_df["normed_date"]).dt.normalize()

static_rvw_003 = static_rvw_002.merge(
    fear_df,
    how="left",
    on="normed_date",
    suffixes=("", "_liq")
)

print(static_rvw_003[["normed_date", "fear_greed"]].head(1))
print(static_rvw_003[["normed_date", "fear_greed"]].tail(1))

In [ ]:

static_rvw_003[f'dumm_fear'] = (static_rvw_003['fear_greed'] < 30).astype(int)
static_rvw_003[f'dumm_greed'] = (static_rvw_003['fear_greed'] > 70).astype(int)


In [ ]:
list_vars = static_rvw_003.columns.tolist()
print(f'variables:{len(list_vars)}')
print(f'observations:{len(static_rvw_003)}')
print(textwrap.fill(", ".join(list_vars), width = 250))

### - Modified win rate: using expected return vs the market

In [ ]:

static_rvw_003['ret_g2'] = (static_rvw_003['pl_g']/static_rvw_003['Capital'])
static_rvw_003['ret_spy'] = (static_rvw_003['SPY_px.1']/static_rvw_003['SPY_px'])-1

static_rvw_003['wins_30'] = (static_rvw_003['ret_g2'] > static_rvw_003['ret_spy'] * static_rvw_003['Beta_30']).astype(int)
static_rvw_003['wins_60'] = (static_rvw_003['ret_g2'] > static_rvw_003['ret_spy'] * static_rvw_003['Beta_60']).astype(int)
static_rvw_003['wins_90'] = (static_rvw_003['ret_g2'] > static_rvw_003['ret_spy'] * static_rvw_003['Beta_90']).astype(int)
static_rvw_003['wins_250'] = (static_rvw_003['ret_g2'] > static_rvw_003['ret_spy'] * static_rvw_003['Beta_250']).astype(int)


In [ ]:
static_rvw_003.head(2)

### - Overlap of different win rates vs standard definition

In [ ]:
features = ['wins_30', 'wins_60', 'wins_90', 'wins_250']

for feature in features:
    print(f"\n--- Probability of 'wins' given '{feature}' ---")
    
    xtab_pct = pd.crosstab(
        index=static_rvw_003['wins'], 
        columns=static_rvw_003[feature],
        normalize='columns'  # Normalizes down the columns
    )
    xtab_formatted = (xtab_pct * 100).round(2).astype(str) + '%'
    
    print(xtab_formatted)

# Numbers below, calculating with the closing Px (not with SPY_px.1 that is the exit px when order was sent, according to Bulat)
# --- Probability of 'wins' given 'wins_30' ---
# wins_30       0       1
# wins                   
# 0        90.39%  12.14%
# 1         9.61%  87.86%

# --- Probability of 'wins' given 'wins_60' ---
# wins_60       0       1
# wins                   
# 0        90.77%  11.75%
# 1         9.23%  88.25%

# --- Probability of 'wins' given 'wins_90' ---
# wins_90       0       1
# wins                   
# 0        90.89%  11.67%
# 1         9.11%  88.33%

# --- Probability of 'wins' given 'wins_250' ---
# wins_250       0       1
# wins                    
# 0         91.01%  11.58%
# 1          8.99%  88.42%

In [ ]:
summary = (
    static_rvw_003.isna().sum()
    .to_frame("missing")
    .assign(
        dtype=static_rvw_003.dtypes,
        non_null=lambda x: len(static_rvw_003) - x["missing"],
        percent_missing=lambda x: x["missing"] / len(static_rvw_003) * 100,
        unique=static_rvw_003.nunique()
    )
)

gc.collect()
print(summary)

In [ ]:
vars_path = model_table_output_path
outname = f"data_dictionary_{datetime.now().strftime('%Y%m%d')}.xlsx"

summary.to_excel(vars_path / outname, index=True, engine='openpyxl')
print(vars_path / outname)

# 3.0 Data Cleaning

### - Data copy 2

In [ ]:
partition_a = static_rvw_003.copy()
print(len(partition_a))
partition_a.head(2)

In [ ]:
summary_df = append_summary(static_rvw_002, "static_rvw_002")
summary_df = append_summary(static_rvw_003, "static_rvw_003")
summary_df = append_summary(partition_a, "partition_a")
print(summary_df)

#              Name     obs ncols
# 0      event_data  893306   117
# 1  static_rvw_001  892373   127
# 2  static_rvw_002  892373   235
# 3  static_rvw_003  892373   244
# 4     partition_a  892373   244

### - Lib import

In [ ]:
# NOTE change for "_37"
import notebook_utils._optimization_functions_37_v2 as opt
importlib.reload(opt)
from notebook_utils._optimization_functions_37_v2 import (optimization_ranges, process_optmz_minmax, optmz_loop_wrap, process_optmz_breaks, optmz_search_with_exclusions, sum_combinations, create_combinations, text_to_dict, outlier_bound, optmz_loop_wrap_with_exclusions,)

In [ ]:
# Main dictionary for time-serie data aggregation
sum_cols = ['mtm_pl', 'entry_pl', 'matched_shares', 'entry_side', 'entry_fees', 'exit_fees', 'exit_shares', 'pl_g', 'pl_n', 'fees']
sum_dict = {key: 'sum' for key in (sum_cols)}

In [ ]:
# Importing optimization master list

vars_path = model_varnames_path
vars_name = 'master_variable_inventory 20260213.xlsx' #Master data dictionary to format columns of backtest

optimization_col_df = pd.read_excel(vars_path / vars_name, sheet_name='optimization', usecols=['clean name', 'optimize', 'group', 'subgroup'])


### - Merging master list of variables to be optimized with new features to create consolidated list

In [ ]:
column_names = partition_a.columns.tolist()
feature_inventory_df = pd.DataFrame(column_names, columns=["clean name"])
print(len(feature_inventory_df))

feature_inventory_df = feature_inventory_df.merge(optimization_col_df, on="clean name", how="left")
print(len(feature_inventory_df))

In [ ]:
# IMPORTANT STEP BECAUSE IT ALLOWS TO REVIEW DATA MANUALLY
# Exporting variable inventory - if proceedure done the same day, no need to chage datetime.now statement
vars_path = DATA_INTERMEDIATE / model
# outname = f"variable_inventory_expanded_{datetime.now().strftime('%Y%m%d')}.xlsx"
outname = f"variable_inventory_expanded_{date_file}.xlsx"
print(vars_path / outname)

# with pd.ExcelWriter(vars_path / outname, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    # feature_inventory_df.to_excel(writer, sheet_name='optimization', index=False)


In [ ]:
# Importing list of target variables to optimize - AFTER REVIEW

vars_name = 'variable_inventory_expanded_20260213.xlsx'

df = pd.read_excel(vars_path / vars_name, sheet_name='optimization', usecols=['clean name', 'optimize', 'group', 'subgroup'])
df = df[df['optimize'] == True][['clean name']].reset_index()

# All variables to optimize for single optimization and best break review
optmz_list_all = df['clean name'].tolist()
print(len(optmz_list_all))
print(textwrap.fill(", ".join(optmz_list_all), width = 250))


# 3.1 Outlier Deletion

In [ ]:
key_vars = ['ATR', 'Acc_vol', 'mtm_pl']

print(len(key_vars))
print(key_vars)

key_consol_dist = pd.DataFrame()

# Loop over each variable and compute the decile summary
for var in key_vars:
    summary = decile_summary(partition_a, var)
    key_consol_dist[var] = summary  # Store the summary in the DataFrame with the variable name as the column

key_consol_dist

### - Initial chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(x='ATR', y='mtm_pl', data=partition_a, ax=axes[0])
sns.scatterplot(x='Acc_vol', y='mtm_pl', data=partition_a, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
# MtM outliers
sample, pnl_cnt = outlier_bound(partition_a, 'mtm_pl', 0.000005, operator.lt)
partition_a['ol_mtm_pl'] = np.where((partition_a['mtm_pl'] < partition_a['mtm_pl'].quantile(0.0000025)), 1,0)
print(sample['mtm_pl'].sort_values(ascending=[False]).head(10))

# Accum Vol outliers
sample, pnl_cnt = outlier_bound(partition_a, 'Acc_vol', 0.999995, operator.gt)
partition_a['ol_vol_acc'] = np.where((partition_a['Acc_vol'] > partition_a['Acc_vol'].quantile(0.999995)), 1,0)
print(sample['Acc_vol'].sort_values(ascending=[False]).head(10))

# ATR outliers
sample, pnl_cnt = outlier_bound(partition_a, 'ATR', 0.99999, operator.gt)
partition_a['ol_ATR'] = np.where((partition_a['ATR'] > partition_a['ATR'].quantile(0.99999)), 1,0)
print(sample['ATR'].sort_values(ascending=[False]).tail(10))

### - Inspecting and deleting outliers

In [ ]:
outliers = [col for col in partition_a.columns if col.startswith('ol_')]
partition_a['ol_all'] = partition_a[outliers].max(axis=1)
ol_all_sum = partition_a['ol_all'].sum()
outliers_obs = partition_a[partition_a['ol_all'] == 1]

# Print the result
print("Sum of 'ol_all':", ol_all_sum)
outliers_obs.head(5)

In [ ]:
partition_a_001 = partition_a[partition_a['ol_all'] == 0]
print(f'new data:{len(partition_a_001)}')
print(f'old data:{len(partition_a)}')

# new data:892356
# old data:892373

### - Final chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(x='ATR', y='mtm_pl', data=partition_a_001, ax=axes[0])
sns.scatterplot(x='Acc_vol', y='mtm_pl', data=partition_a_001, ax=axes[1])
plt.tight_layout()
plt.show()


In [ ]:
summary_df = append_summary(partition_a_001, "partition_a_001")
print(summary_df)

# 3.2 Test/Train Data splits

### - INS/OOS

In [ ]:
break_pct = 0.85
sample = int(round(len(partition_a_001) * break_pct, 0))
sample_breakdate = partition_a_001.iloc[sample]['normed_date']
print(f"Break date: {sample_breakdate}")

partition_ins = partition_a_001[partition_a_001['normed_date'] < sample_breakdate]
partition_oos = partition_a_001[partition_a_001['normed_date'] >= sample_breakdate]

print(f"obs: {len(partition_ins)}")
print(partition_ins['normed_date'].head(1))
print(partition_ins['normed_date'].tail(1))
print("")
print(f"obs: {len(partition_oos)}")
print(partition_oos['normed_date'].head(1))
print(partition_oos['normed_date'].tail(1))


# Break date: 2024-09-11 00:00:00
# obs: 758364
# 0   2013-09-24
# 758379   2024-09-10

# obs: 133992
# 758380   2024-09-11
# 892372   2026-01-14

### - 80/20 split for INS

In [ ]:
break_pct = 0.80
sample = int(round(len(partition_ins) * break_pct, 0))
sample_breakdate = partition_ins.iloc[sample]['normed_date']
print(sample_breakdate)

partition_ins_80 = partition_ins[partition_ins['normed_date'] < sample_breakdate]
partition_ins_20 = partition_ins[partition_ins['normed_date'] >= sample_breakdate]

print(f"obs: {len(partition_ins_80)}")
print(partition_ins_80['normed_date'].head(1))
print(partition_ins_80['normed_date'].tail(1))
print("")
print(f"obs: {len(partition_ins_20)}")
print(partition_ins_20['normed_date'].head(1))
print(partition_ins_20['normed_date'].tail(1))

# 2022-09-20 00:00:00
# obs: 606395
# 0   2013-09-24
# 606399   2022-09-19

# obs: 151969
# 606400   2022-09-20
# 758379   2024-09-10

In [ ]:
# INS/OOS
summary_df = append_summary(partition_ins, "partition_ins")
summary_df = append_summary(partition_oos, "partition_oos")
# INS (train/test)
summary_df = append_summary(partition_ins_20, "partition_ins_20")
summary_df = append_summary(partition_ins_80, "partition_ins_80")
print(summary_df)

#          Name     obs ncols
# 0        event_data  893306   117
# 1    static_rvw_001  892373   127
# 2    static_rvw_002  892373   235
# 3    static_rvw_003  892373   244
# 4       partition_a  892373   244
# 5   partition_a_001  892356   248
# 6     partition_ins  758364   248
# 7     partition_oos  133992   248
# 8  partition_ins_20  151969   248
# 9  partition_ins_80  606395   248

# 3.3 Correlation analysis

### - Correlations with P/L

In [ ]:
corr_list = ['mtm_pl'] + optmz_list_all
correlation_matrix = partition_ins_80[corr_list].corr()

corr_mtmpl = (
    correlation_matrix[['mtm_pl']]
    .assign(mtm_pl_abs=lambda df: df['mtm_pl'].abs())
    .sort_values(by='mtm_pl_abs', ascending=False)
)

top_corr_mtmpl = corr_mtmpl[1:26]
top_corr_mtmpl_list = top_corr_mtmpl.index.tolist()
print(len(top_corr_mtmpl_list))
print(textwrap.fill(", ".join(top_corr_mtmpl_list), width = 250))


### - Highest correlated variables among themselves

In [ ]:
# Selection correlations greater than 79.99% - only positive correlations as negative ones are regarded as diversifiers
ref_corr = 0.7999

high_corr_counts = (correlation_matrix > ref_corr).sum(axis=0) - 1  # subtract 1 to exclude correlation of the column with itself
filtered_high_corr_counts = high_corr_counts[high_corr_counts > 3].sort_values(ascending=False)
high_corr_list = filtered_high_corr_counts.index.tolist()
print(len(high_corr_list))
print(textwrap.fill(", ".join(high_corr_list), width = 250))


### - High correlation between variables highly correlated with P/L

In [ ]:
# High correlation between variables highly correlated with P/L
top_correlation_matrix = partition_ins_80[top_corr_mtmpl_list].corr()

ref_corr = 0.7999
top_high_corr_counts = (top_correlation_matrix > ref_corr).sum(axis=0) - 1  # subtract 1 to exclude correlation of the column with itself
filtered_top_high_corr_counts = top_high_corr_counts[high_corr_counts > 3].sort_values(ascending=False)

top_high_corr_list = filtered_top_high_corr_counts.index.tolist()
print(len(top_high_corr_list))
print(textwrap.fill(", ".join(top_high_corr_list), width = 250))

# ret_EMA_50_2, ret_EMA_50, ret_EMA_100, ret_EMA_100_2, ret_EMA_200_2, ret_EMA_200, dist_SPY_px_SPY_ema_50, ret_EMA_20_2, ret_EMA_20, dist_SPY_px_SPY_ema_20

### - Top 25 variables (high corr with P/L) and cross-correlations > 80%

In [ ]:
# Create the "consolidated" column

cross_corr = 0.7999
top_correlation_matrix['consolidated'] = [
    [col for col in top_correlation_matrix.columns if top_correlation_matrix.loc[row, col] > cross_corr]
    for row in top_correlation_matrix.index
]

# Filter out rows where "consolidated" is empty
result_df = top_correlation_matrix[['consolidated']].loc[~top_correlation_matrix['consolidated'].apply(lambda x: len(x) == 0)]

# Cleaning data to remove repeated index names on "consolidated"
result_df['consolidated'] = [
    [item for item in row if str(index_name) not in str(item)]
    for index_name, row in zip(result_df.index, result_df['consolidated'])
]

print(result_df)


# 4.0 Pre ML-Optimization

### - Removing obs from Pre-optimization (I): For INS data (80%-train)

In [ ]:
# Removing observations with wide spreads (greated than 20 bps)
# partition_ins_80_001 = partition_ins_80[(partition_ins_80['ATR'] >= 0.5) & (partition_ins_80['Beta'] >= 0.50) & (partition_ins_80['AVOL'] >= 2.5e6)]
partition_ins_80_001 = partition_ins_80[(partition_ins_80['ATR'] >= 0.0) & (partition_ins_80['Beta_250'] >= 0.00) & (partition_ins_80['AVOL'] >= 0.0)]
print(f'new data:{len(partition_ins_80_001)}')
print(f'old data:{len(partition_ins_80)}')

# new data:601838
# old data:606395

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(x='ATR', y='mtm_pl', data=partition_ins_80_001, ax=axes[0])
sns.scatterplot(x='Acc_vol', y='mtm_pl', data=partition_ins_80_001, ax=axes[1])
plt.tight_layout()
plt.show()


### - Removing obs from Pre-optimization (I): For INS data (20%-test)

In [ ]:
partition_ins_20_001 = partition_ins_20[(partition_ins_20['ATR'] >= 0.0) & (partition_ins_20['Beta_250'] >= 0.00) & (partition_ins_20['AVOL'] >= 0.0)]
print(f'new data:{len(partition_ins_20_001)}')
print(f'old data:{len(partition_ins_20)}')

# new data:151694
# old data:151969

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.scatterplot(x='ATR', y='mtm_pl', data=partition_ins_20_001, ax=axes[0])
sns.scatterplot(x='Acc_vol', y='mtm_pl', data=partition_ins_20_001, ax=axes[1])
plt.tight_layout()
plt.show()


### - Dashboard for raw INS (cleaned) data

In [ ]:
strat_ins_001, ins_bydate_001 = dashboard(partition_ins_80_001, comm=3.5, bpower=1000000, mod_id='ins_raw', long_short=1, dash_type='short')

xls_outpath = model_xls_output_path
dataname = 'sancocho_long_ml expanded '
outname = f"- strat_ins_01 {datetime.now().strftime('%Y%m%d')}.xlsx"

# strat_ins_001.to_excel(xls_outpath / f"{dataname}{outname}", index = True, engine='openpyxl')
print(xls_outpath / f"{dataname}{outname}") 

# 4.1 ML Optimization

### - Missing observations

In [ ]:
# Infis first
numeric_df = partition_ins_80_001.select_dtypes(include=[np.number])
inf_cols = numeric_df.columns[np.isinf(numeric_df).any()]
print("Columns with Inf/-Inf values:", inf_cols.tolist())
# NaN second
nan_cols = partition_ins_80_001.columns[partition_ins_80_001.isna().any()]
print("Columns with NaN values:", nan_cols.tolist())

# Summary
_df_ = partition_ins_80_001
# NaN counts (all columns)
nan_count = _df_.isna().sum()
nan_count = nan_count[nan_count > 0]

# Inf/-Inf counts (numeric columns only)
numeric_df = _df_.select_dtypes(include=[np.number])
inf_count = np.isinf(numeric_df).sum()
inf_count = inf_count[inf_count > 0]

# Combine into one summary DataFrame
summary_df = pd.DataFrame({
    'NaN Count': nan_count,
    'Inf Count': inf_count
}).fillna(0).astype(int)

print(summary_df)

### - Deleting missing observations (only for optimization features)

In [ ]:
partition_ins_80_002 = partition_ins_80_001.dropna(subset=["Agg_Gamma_1"])
partition_ins_20_002 = partition_ins_20_001.dropna(subset=["Agg_Gamma_1"])

In [ ]:
numeric_df = partition_ins_80_002.select_dtypes(include=[np.number])
inf_cols = numeric_df.columns[np.isinf(numeric_df).any()]
print("Columns with Inf/-Inf values:", inf_cols.tolist())
# NaN second
nan_cols = partition_ins_80_002.columns[partition_ins_80_002.isna().any()]
print("Columns with NaN values:", nan_cols.tolist())

# Summary
_df_ = partition_ins_80_002
# NaN counts (all columns)
nan_count = _df_.isna().sum()
nan_count = nan_count[nan_count > 0]

# Inf/-Inf counts (numeric columns only)
numeric_df = _df_.select_dtypes(include=[np.number])
inf_count = np.isinf(numeric_df).sum()
inf_count = inf_count[inf_count > 0]

# Combine into one summary DataFrame
summary_df = pd.DataFrame({
    'NaN Count': nan_count,
    'Inf Count': inf_count
}).fillna(0).astype(int)

print(summary_df)

In [ ]:
summary_df = append_summary(partition_ins_80_001, "partition_ins_80_001")
summary_df = append_summary(partition_ins_80_002, "partition_ins_80_002")
summary_df = append_summary(partition_ins_20_001, "partition_ins_20_001")
summary_df = append_summary(partition_ins_20_002, "partition_ins_20_002")
print(summary_df)

### - ML Train/Test splits

In [ ]:
short_ml_list = ['ATR', 
                'Daily_CMF', 
                'RSI', 
                'SPY_atr',
                'fear_greed',
                'dumm_fear',
                'dist_SPY_px_SPY_ema_50',
                'dist_SPY_px_SPY_ema_8',
                'SPY_rsi',
                'week_day_cos',
                'PCA_Index_full',
                'SPY_Spot_Gamma_st_dev',
                'dist_SPY_px_SPY_close',
                'momo_px_2',
                'momo_px_8',
                'dist_Last_px_Prev_close_2',
                'dist_Last_px_prev_low',
                'dist_Last_px_prev_open',
                'dist_Last_px_prev_close',
                'dumm_Var_pred_2'
                ]


In [ ]:
# Reference is wins_30 (excess return over the 30-day Beta)
# ml_vars = optmz_list_all
ml_vars = short_ml_list

X_train = partition_ins_80_002[ml_vars]
# y_train  = partition_ins_80_002['wins']
# y_train  = partition_ins_80_002['wins_30']
y_train  = partition_ins_80_002['wins_60']

X_test = partition_ins_20_002[ml_vars]
# y_test  = partition_ins_20_002['wins']
# y_test  = partition_ins_20_002['wins_30']
y_test  = partition_ins_20_002['wins_60']

# Only dependent variable needed for OOS as we will fit the optimal number of features of X, which is calculated later
# partition_oos

# y_oos  = partition_oos['wins']
# y_oos  = partition_oos['wins_30']
y_oos  = partition_oos['wins_60']

print(f'train: {len(y_train)}')
print("Train Win %", np.mean(y_train))
print("")
print(f'test: {len(y_test)}')
print("Test Win %", np.mean(y_test))
print("")
print(f'test: {len(y_oos)}')
print("Test Win %", np.mean(y_oos))


### - Hyperparameter tunning (*No need to run every time) - TAKES TOO MUCH TIME TO RUN!

In [ ]:
import notebook_utils._grid_search_37_v2 as grid
importlib.reload(grid)
from notebook_utils._grid_search_37_v2 import (build_base_xgb, param_distributions, randomized_search_on_sample, final_refit_with_early_stopping, tune_xgb_fast_compatible,)


#### - A1: longest run with all the data - full grid search

#### - E1: Faster random grid search with full data and selected (high corr) features

In [ ]:
# Hypertunning on data with all features (about 150)
model, best = tune_xgb_fast_compatible(X_train, y_train, X_test, y_test, search_rows=200_000, n_iter=60, cv_splits=5)
print("Best params:", best)
print("Best ntree limit:", getattr(model, "best_ntree_limit", None))
print("Best iteration:", getattr(model, "best_iteration", None))

# Test recall: 0.6926031033362983
#               precision    recall  f1-score   support

#            0       0.52      0.34      0.41     70311
#            1       0.51      0.69      0.59     70827

#    micro avg       0.52      0.52      0.52    141138
#    macro avg       0.52      0.52      0.50    141138
# weighted avg       0.52      0.52      0.50    141138

# Best params: {'colsample_bytree': 0.8170160922219597, 'gamma': 0.305288446103256, 'max_depth': 10, 'min_child_weight': 0.11502956321912727, 'n_estimators': 400, 'reg_alpha': 0.0016593890383815794, 'reg_lambda': 77.99059237255672, 'subsample': 0.7572390898667042}
# Best ntree limit: 555
# Best iteration: 554

# NEW
# Best params: {'colsample_bytree': 0.8080272084711243, 'gamma': 0.32802616760596776, 'max_depth': 10, 'min_child_weight': 0.01334299028518794, 'n_estimators': 400, 'reg_alpha': 1.6271360716645864, 'reg_lambda': 0.629530148451613, 'subsample': 0.7580600944007257}
# Best ntree limit: 233
# Best iteration: 232

In [ ]:
# Two options: 
# Run with faster algo

# OLD
# best_params = {'colsample_bytree': 0.82, 'gamma': 0.31, 'max_depth': 10, 'min_child_weight': 0.115, 'n_estimators': 400, 'reg_alpha': 0.002, 'reg_lambda': 78, 'subsample': 0.76}

# NEW
# best_params = {'colsample_bytree': 0.80, 'gamma': 0.33, 'max_depth': 10, 'min_child_weight': 0.013, 'n_estimators': 400, 'reg_alpha': 1.63, 'reg_lambda': 0.36, 'subsample': 0.76}

# re-NEW (wins)
best_params = {'colsample_bytree': 0.81, 'gamma': 0.19, 'max_depth': 8, 'min_child_weight': 0.12, 'n_estimators': 400, 'reg_alpha': 6.7, 'reg_lambda': 0.67, 'subsample': 0.93}

# Full grid search (used in saved pickle)
# best_params = {'colsample_bytree': 0.7, 'gamma': 0.1, 'max_depth': 20, 'n_estimators': 100, 'reg_alpha': 1, 'subsample': 0.9}

# Assume you've already run GridSearchCV and found the best parameters
# best_params = grid_search.best_params_

# Step 1: Create a new XGB with the best parameters
best_xgb = xgb.XGBClassifier(**best_params,
                            learning_rate=0.1,
                            random_state=42,
                            eval_metric='logloss',
                            objective="binary:logistic",
                            use_label_encoder=False,
                            tree_method="hist",
                            n_jobs=-1,
                            )


In [ ]:
# Step 2: Train the model on your training data
best_xgb.fit(X_train, y_train)

# Step 3: Evaluate or use the model
pred_train = best_xgb.predict(X_train)
pred_test = best_xgb.predict(X_test)

print(classification_report(y_train,pred_train))
print(confusion_matrix(y_train,pred_train))

print(classification_report(y_test,pred_test))
print(confusion_matrix(y_test,pred_test))

#               precision    recall  f1-score   support

#            0       0.76      0.75      0.75    254220
#            1       0.75      0.76      0.75    253007

#    micro avg       0.75      0.75      0.75    507227
#    macro avg       0.75      0.75      0.75    507227
# weighted avg       0.75      0.75      0.75    507227

# [[190407  63813]
#  [ 61569 191438]]
#               precision    recall  f1-score   support

#            0       0.53      0.36      0.43     70311
#            1       0.52      0.68      0.59     70827

#    micro avg       0.52      0.52      0.52    141138
#    macro avg       0.52      0.52      0.51    141138
# weighted avg       0.52      0.52      0.51    141138

# [[25451 44860]
#  [22968 47859]]

In [ ]:
print("ML Predictions from Train data")
print("Obs:", len(pred_train))
print("Mean:", np.mean(pred_train))
print("P/L+", np.sum(pred_train == 1))

print("")
print("ML Predictions from Test data")
print("Obs:", len(pred_test))
print("Mean:", np.mean(pred_test))
print("P/L+", np.sum(pred_test == 1))

# ML Predictions from Train data
# Obs: 507227
# Mean: 0.503228337608211
# P/L+ 255251

# ML Predictions from Test data
# Obs: 141138
# Mean: 0.6569385991015885
# P/L+ 92719

### - Feature optimization (* No need to run every time)

In [ ]:
# Applying RF estimator to the RFECV to deal with memory allocation isssue

cv_split = TimeSeriesSplit(max_train_size=None, n_splits=3)
rfecv = RFECV(estimator=best_xgb, step=10, min_features_to_select=2, cv=cv_split, scoring='recall', n_jobs=-1)

# rf_selector_model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
# rfecv = RFECV(estimator=rf_selector_model, step=5, min_features_to_select=2, cv=cv_split, scoring='recall', n_jobs=-1)


#### - A2 Feature optimization - mini data

In [ ]:

# rfecv.fit(X_train_mini, y_train_mini)

# # Number of selected features
# print(f"Optimal number of features: {rfecv.n_features_}")
# # print(f"Optimal features: {list(rfecv.get_feature_names_out())}")

# selected_mask = rfecv.support_
# selected_features = X_train_mini.columns[selected_mask]
# print(f"Optimal features: {list(selected_features)}")


#### - B2: Feature optimization with full data and ALL features

In [ ]:
rfecv.fit(X_train, y_train)

# Number of selected features
print(f"Optimal number of features: {rfecv.n_features_}")
# print(f"Optimal features: {list(rfecv.get_feature_names_out())}")

selected_mask = rfecv.support_
selected_features = X_train.columns[selected_mask]
print(f"Optimal features: {list(selected_features)}")


In [ ]:

plt.plot(range(1, len(rfecv.grid_scores_) + 1), rfecv.grid_scores_)
plt.xlabel("Number of features selected")
plt.ylabel("Cross-validated recall")
plt.title("RFECV - Recall vs Number of Features")
plt.grid(True)
plt.show()


In [ ]:
feature_ranks = pd.Series(rfecv.ranking_, index=X_train.columns).sort_values()
print(feature_ranks)


In [ ]:
selected_features = X_train.columns[rfecv.support_]
importances = rfecv.estimator_.feature_importances_
pd.Series(importances, index=selected_features).sort_values(ascending=False)


### - Running best specification (per RFECV)

In [ ]:
# rfecv_vars =  selected_features
rfecv_vars = ['PCA_Index_full', 'SPY_Spot_Gamma_st_dev', 'SPY_atr', 'SPY_rsi', 'dist_SPY_px_SPY_ema_8', 'dist_SPY_px_SPY_close', 'dist_SPY_px_SPY_ema_50', 'fear_greed', 'dist_Last_px_prev_close', 'dist_Last_px_Prev_close_2']
# rfecv_vars = ['ATR', 'Daily_CMF', 'RSI', 'dist_SPY_px_SPY_ema_8', 'SPY_Spot_Gamma_st_dev', 'dist_SPY_px_SPY_close', 'dist_Last_px_Prev_close_2', 'dist_Last_px_prev_low', 'dist_Last_px_prev_open', 'dist_Last_px_prev_close']

X_train_rfecv = partition_ins_80_002[rfecv_vars]
X_test_rfecv = partition_ins_20_002[rfecv_vars]
X_oos_rfecv = partition_oos[rfecv_vars]


In [ ]:
X_train_rfecv.head()

In [ ]:
# best_params = {'colsample_bytree': 0.7, 'gamma': 0.1, 'max_depth': 20, 'n_estimators': 100, 'reg_alpha': 1, 'subsample': 0.9}

best_rfecv_xgb = xgb.XGBClassifier(**best_params, 
                                    learning_rate=0.1, 
                                    random_state=42, 
                                    eval_metric='logloss',
                                    use_label_encoder=False,
                                    objective="binary:logistic",
                                    tree_method="hist",
                                    n_jobs=-1,
                                )

# Fitting/evaluating: INS Train
best_rfecv_xgb.fit(X_train_rfecv, y_train)

# Predicting/evaluating: INS Train
pred_train_rfecv = best_rfecv_xgb.predict(X_train_rfecv)
yhat_train = best_rfecv_xgb.predict_proba(X_train_rfecv)
yhat_train_1 = yhat_train[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_train_1))
print("Mean:", np.mean(yhat_train_1))

# Predicting/evaluating: INS Test
pred_test_rfecv = best_rfecv_xgb.predict(X_test_rfecv)
yhat_test = best_rfecv_xgb.predict_proba(X_test_rfecv)
yhat_test_1 = yhat_test[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_test_1))
print("Mean:", np.mean(yhat_test_1))

# Predicting/evaluating: OOS
pred_oos_rfecv = best_rfecv_xgb.predict(X_oos_rfecv)
yhat_oos = best_rfecv_xgb.predict_proba(X_oos_rfecv)
yhat_oos_1 = yhat_oos[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_oos_1))
print("Mean:", np.mean(yhat_oos_1))

# Obs: 697844
# Mean: 0.5109056
# Obs: 158623
# Mean: 0.52697307
# Obs: 274637
# Mean: 0.52232665

In [ ]:
pred_train_rfecv

In [ ]:
print(classification_report(y_train,pred_train_rfecv))
print(confusion_matrix(y_train,pred_train_rfecv))

print(classification_report(y_test,pred_test_rfecv))
print(confusion_matrix(y_test,pred_test_rfecv))

print(classification_report(y_oos,pred_oos_rfecv))
print(confusion_matrix(y_oos,pred_oos_rfecv))

#               precision    recall  f1-score   support

#            0       0.51      0.42      0.46    134916
#            1       0.52      0.61      0.56    139721

#    micro avg       0.51      0.51      0.51    274637
#    macro avg       0.51      0.51      0.51    274637
# weighted avg       0.51      0.51      0.51    274637

# [[56403 78513]
#  [54838 84883]]


In [ ]:
# Just to confirm list of features
print(best_rfecv_xgb.get_booster().feature_names)

In [ ]:

# ROC for Train
fpr_train, tpr_train, _ = roc_curve(y_train, yhat_train_1)
roc_auc_train = auc(fpr_train, tpr_train)

# ROC for Test
fpr_test, tpr_test, _ = roc_curve(y_test, yhat_test_1)
roc_auc_test = auc(fpr_test, tpr_test)

# Plot both
plt.figure(figsize=(6, 5))
plt.plot(fpr_train, tpr_train, label=f'Train ROC (AUC = {roc_auc_train:.2f})')
plt.plot(fpr_test, tpr_test, label=f'Test ROC (AUC = {roc_auc_test:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost (Train vs Test)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

### - Saving specification for KITE load* (saving only once)

In [ ]:
model = 'sancocho'
loc = model_ml_output_path
filename = f"xgb_{model}_expanded_long_v1.pkl"

# ML object for expanded list of tickers
ml_path = loc / filename
with open(ml_path, 'wb') as f:
    pickle.dump(best_rfecv_xgb, f)
print(f"Model saved at: {ml_path}")


### - Data copy 3

In [ ]:
# Note:
# - Step 001 deletions: ATR, Beta, AVOL
# - Step 002 deletions: missing obs (51038...via agg_gamma)

partition_ins_80_003 = partition_ins_80_002.copy()
partition_ins_20_003 = partition_ins_20_002.copy()
partition_oos_001 = partition_oos.copy()

### - Merging predicted values to INS-80%/20% and OOS

In [ ]:
if len(partition_ins_80_003) == len(yhat_train_1):
    partition_ins_80_003['yhat_train_1'] = yhat_train_1
else:
    raise ValueError("Length mismatch between DataFrame and NumPy array.")

if len(partition_ins_20_003) == len(yhat_test_1):
    partition_ins_20_003['yhat_train_1'] = yhat_test_1
else:
    raise ValueError("Length mismatch between DataFrame and NumPy array.")
partition_ins_20_003.reset_index(drop=True, inplace=True)

if len(partition_oos_001) == len(yhat_oos_1):
    partition_oos_001['yhat_train_1'] = yhat_oos_1
else:
    raise ValueError("Length mismatch between DataFrame and NumPy array.")
partition_oos_001.reset_index(drop=True, inplace=True)

In [ ]:
partition_oos_001.head()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
sns.scatterplot(x='yhat_train_1', y='mtm_pl', data=partition_ins_80_003, ax=axes[0])
sns.scatterplot(x='RSI', y='mtm_pl', data=partition_ins_80_003, ax=axes[1])
sns.scatterplot(x='fear_greed', y='mtm_pl', data=partition_ins_80_003, ax=axes[2])
sns.scatterplot(x='fear_greed', y='yhat_train_1', data=partition_ins_80_003, ax=axes[3])


### - Final data summary

In [ ]:
summary_df = append_summary(partition_ins_80_003, "partition_ins_80_003")
summary_df = append_summary(partition_ins_20_003, "partition_ins_20_003")
summary_df = append_summary(partition_oos_001, "partition_oos_001")
print(summary_df)


### - Saving final data

In [ ]:
# !{sys.executable} -m pip install "pyarrow==10.0.1"

In [ ]:
loc = model_intermediate_path
outdir = Path(loc)
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
dfs = {
    "partition_ins_80_003": partition_ins_80_003,
    "partition_ins_20_003": partition_ins_20_003,
    "partition_oos_001":    partition_oos_001,
}

for name, df in dfs.items():
    df.to_parquet(outdir / f"{name}.parquet",
                  engine="pyarrow",   # if missing, use engine="fastparquet"
                  index=False,
                  compression="zstd") # if unavailable, try "snappy"
    
# partition_ins_80_003.to_parquet(outdir / "partition_ins_80_003.parquet", engine="pyarrow", index=False, compression="zstd")
# partition_ins_20_003.to_parquet(outdir / "partition_ins_20_003.parquet", engine="pyarrow", index=False, compression="zstd")
# partition_oos_001.to_parquet(   outdir / "partition_oos_001.parquet",    engine="pyarrow", index=False, compression="zstd")

# Loading example!
# load_df = lambda name, columns=None: pd.read_parquet(Path(loc) / f"{name}.parquet", columns=columns)
# usage:
# ins80 = load_df("partition_ins_80_003")
# ins20_small = load_df("partition_ins_20_003", columns=["id","x"])
# oos = load_df("partition_oos_001")

# one-liner saver (pyarrow)
# save_df = lambda df, name: df.to_parquet(Path(loc) / f"{name}.parquet", engine="pyarrow", index=False, compression="zstd")
# usage:
# save_df(partition_ins_80_003, "partition_ins_80_003")
# save_df(partition_ins_20_003, "partition_ins_20_003")
# save_df(partition_oos_001,    "partition_oos_001")



# 5.0 Non-ML Optimization

### - Defining optimization ranges and steps

In [ ]:
partition_ins_80_003.head(2)

In [ ]:

optmz_list_all_plus = ['ATR', 'Beta_30', 'Beta_60', 'Beta_90','Daily_CMF', 'RSI', 'St_dev_20', 'St_dev_5', 'St_dev_60', 'SPY_atr', 'SPY_rsi', 'SPY_st_dev', 'SPY_Agg_Gamma_z_score', 'SPY_Spot_Gamma_z_score', 'momo_px_2', 'momo_px_9',
                       'dist_Last_px_Arimax_pred_1',  'dist_Last_px_Arimax_pred_2', 'dist_Last_px_EMA_100', 'dist_Last_px_EMA_20', 'dist_Last_px_EMA_200', 'dist_Last_px_EMA_50', 'dist_Last_px_EMA_8', 'dist_Last_px_Prev_close_10',
                        'dist_Last_px_Prev_close_2', 'dist_Last_px_Prev_close_3', 'dist_Last_px_Prev_close_4', 'dist_Last_px_Prev_close_5', 'dist_Last_px_Prev_close_6', 'dist_Last_px_Prev_close_7', 'dist_Last_px_Prev_close_8',
                        'dist_Last_px_Prev_close_9', 'dist_Last_px_Prev_vwap', 'dist_Last_px_Var_pred_1', 'dist_Last_px_Var_pred_2', 'dist_Last_px_prev_close', 'dist_Last_px_prev_high', 'dist_Last_px_prev_low', 'dist_Last_px_prev_open',
                        'dist_SPY_px_SPY_close', 'dist_SPY_px_SPY_ema_20', 'dist_SPY_px_SPY_ema_200', 'dist_SPY_px_SPY_ema_50', 'dist_SPY_px_SPY_ema_8', 'dist_SPY_px_SPY_prev_close', 'Premkt_vol_rat', 'prev_askvol_rat', 'prev_vol_rat',
                        'Acc_vol_rat', 'week_day_sin','week_day_cos', 'ret_Prev_close_2', 'ret_Prev_close_3', 'ret_Prev_close_4', 'ret_Prev_close_5', 'ret_Prev_close_6', 'ret_Prev_close_7', 'ret_Prev_close_8', 'ret_Prev_close_9', 
                        'ret_Prev_close_10', 'pos_ret', 'pos_pct', 'ret_EMA_8', 'ret_EMA_20', 'ret_EMA_50', 'ret_EMA_100', 'ret_EMA_200', 'pos_ma_ret', 'pos_ma_pct', 'PCA_ScaledIndex_ma50', 'PCA_Index_full', 'fear_greed', 'yhat_train_1'
                        ]

print(f'Optimizable variables: {len(optmz_list_all)}')
print(f'Predicted values added: {len(optmz_list_all_plus)}')

In [ ]:
drop_manual = []
all_ranges, var_stats = optimization_ranges(partition_ins_80_003, optmz_list_all_plus, drop_manual, 20)
all_ranges.drop(all_ranges[all_ranges['steps'] == 0].index, inplace=True)

all_ranges.tail(5)

In [ ]:
# drop_multi = lambda optmz_list_all, prefixes=('Is_', 'momo_', 'dumm'): [n for n in optmz_list_all if re.search(r'^(?:' + '|'.join(map(re.escape, prefixes)) + r')', n) is None]
# filtered_3 = drop_multi(optmz_list_all, prefixes=('Is_', 'momo_', 'dumm'))

### - Data Export

In [ ]:
# Data exports for review
xls_outpath = model_xls_output_path
outname = f"optmz_consol_stats {datetime.now().strftime('%Y%m%d')} v1.xlsx"
var_stats.to_excel(xls_outpath / outname, index = True, engine='openpyxl')
print(xls_outpath / outname)

outname = f"optmz variables 10-90 {datetime.now().strftime('%Y%m%d')} 1.xlsx"
all_ranges.to_excel(xls_outpath / outname, index = True, engine='openpyxl')

print(xls_outpath / outname)

## 5.1 Optimization of single variables for break discovery

In [ ]:
# Create the dictionary with the index as the key and ['min', 'max'] columns as the values
target_vars = {index: (row[0.1], row[0.9], row['steps']) for index, row in all_ranges.iterrows()}
print(f'Target variables: {len(target_vars)}')

# Pass target dictionary to criteria dictionary that will be used by search algorithm
crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
print(f'Search criteria: {len(crit_lst)}')

### - Single variables, positive (>=)

In [ ]:
## TESTING ALL INDIVIDUAL TARGETS
# Applying list of criteria - 238635 max capital used in all data
crit_lst = [{key: (j, '>=')} for key, (element1, element2, element3) in target_vars.items() for j in np.arange(element1, element2, element3)]
single_pos0, optmz, optmz_bydate = optmz_loop_wrap(partition_ins_80_003, crit_lst, 1000000)


# Post-processing consolidates results (selecting first and rd from bottom as a low bar)
# 'extrm' stands for extreme values (first and nth from last: 2 rows per variable)
# 'all' stands for all values (20 rows per variable)
single_pos_extrm, single_pos_all = process_optmz_minmax(single_pos0, 'Sharpe ratio', -3)
gc.collect()

print(f'63 variables x 20 steps: {len(single_pos_all)}')
print(f'63 variables x 2 extreme values: {len(single_pos_extrm)}')

In [ ]:
outname = f"- optimization all result - pos {datetime.now().strftime('%Y%m%d')} v1.xlsx"
# single_pos_all.to_excel(xls_outpath / f"{dataname}{outname}", index = True, engine='openpyxl')


### - Lib import

In [ ]:
import notebook_utils._fast_optimization_v1 as fast_opt
import notebook_utils._bayesian_optimization_v1 as bayes_opt

importlib.reload(fast_opt)
importlib.reload(bayes_opt)

from notebook_utils._fast_optimization_v1 import greedy_threshold_search
from notebook_utils._bayesian_optimization_v1 import bayesian_greedy_threshold_search

## 5.X Fast Optimization

### - With all features

In [ ]:

df = partition_ins_80_003
feats = [
    "ATR",'Beta_30', 'Beta_60', 'Beta_90', "Daily_CMF","RSI","St_dev_20","St_dev_5","St_dev_60", "SPY_atr","SPY_rsi","SPY_st_dev","SPY_Agg_Gamma_z_score","SPY_Spot_Gamma_z_score", "dist_Last_px_Arimax_pred_1","dist_Last_px_Arimax_pred_2","dist_Last_px_EMA_100", 
    "dist_Last_px_EMA_20","dist_Last_px_EMA_200","dist_Last_px_EMA_50","dist_Last_px_EMA_8", "dist_Last_px_Prev_close_10","dist_Last_px_Prev_close_2","dist_Last_px_Prev_close_3", "dist_Last_px_Prev_close_4","dist_Last_px_Prev_close_5","dist_Last_px_Prev_close_6",
    "dist_Last_px_Prev_close_7","dist_Last_px_Prev_close_8","dist_Last_px_Prev_close_9", "dist_Last_px_Prev_vwap","dist_Last_px_Var_pred_1","dist_Last_px_Var_pred_2", "dist_Last_px_prev_close","dist_Last_px_prev_high","dist_Last_px_prev_low", 
    "dist_Last_px_prev_open","dist_SPY_px_SPY_close","dist_SPY_px_SPY_ema_20","dist_SPY_px_SPY_ema_200","dist_SPY_px_SPY_ema_50","dist_SPY_px_SPY_ema_8",  "dist_SPY_px_SPY_prev_close","Premkt_vol_rat","prev_askvol_rat","prev_vol_rat",'momo_px_2', 'momo_px_9', 
    "Acc_vol_rat","week_day_sin","week_day_cos","ret_Prev_close_2","ret_Prev_close_3", "ret_Prev_close_4","ret_Prev_close_5","ret_Prev_close_6","ret_Prev_close_7", "ret_Prev_close_8","ret_Prev_close_9","ret_Prev_close_10","pos_ret","pos_pct", 
    "ret_EMA_8","ret_EMA_20","ret_EMA_50","ret_EMA_100","ret_EMA_200","pos_ma_ret", "pos_ma_pct","PCA_ScaledIndex_ma50","PCA_Index_full","fear_greed","yhat_train_1",
        ]
res = greedy_threshold_search(
    df, feats,
    n_quantiles=24, 
    max_features=6, 
    improvement_eps=0.02,
    adaptive_tails=True, 
    min_count_per_side=200
)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - Excluding ML-probability

In [ ]:
feats = [
    "ATR",'Beta_30', 'Beta_60', 'Beta_90', "Daily_CMF","RSI","St_dev_20","St_dev_5","St_dev_60", "SPY_atr","SPY_rsi","SPY_st_dev","SPY_Agg_Gamma_z_score","SPY_Spot_Gamma_z_score", "dist_Last_px_Arimax_pred_1","dist_Last_px_Arimax_pred_2","dist_Last_px_EMA_100", 
    "dist_Last_px_EMA_20","dist_Last_px_EMA_200","dist_Last_px_EMA_50","dist_Last_px_EMA_8", "dist_Last_px_Prev_close_10","dist_Last_px_Prev_close_2","dist_Last_px_Prev_close_3", "dist_Last_px_Prev_close_4","dist_Last_px_Prev_close_5","dist_Last_px_Prev_close_6",
    "dist_Last_px_Prev_close_7","dist_Last_px_Prev_close_8","dist_Last_px_Prev_close_9", "dist_Last_px_Prev_vwap","dist_Last_px_Var_pred_1","dist_Last_px_Var_pred_2", "dist_Last_px_prev_close","dist_Last_px_prev_high","dist_Last_px_prev_low", 
    "dist_Last_px_prev_open","dist_SPY_px_SPY_close","dist_SPY_px_SPY_ema_20","dist_SPY_px_SPY_ema_200","dist_SPY_px_SPY_ema_50","dist_SPY_px_SPY_ema_8",  "dist_SPY_px_SPY_prev_close","Premkt_vol_rat","prev_askvol_rat","prev_vol_rat",'momo_px_2', 'momo_px_9', 
    "Acc_vol_rat","week_day_sin","week_day_cos","ret_Prev_close_2","ret_Prev_close_3", "ret_Prev_close_4","ret_Prev_close_5","ret_Prev_close_6","ret_Prev_close_7", "ret_Prev_close_8","ret_Prev_close_9","ret_Prev_close_10","pos_ret","pos_pct", 
    "ret_EMA_8","ret_EMA_20","ret_EMA_50","ret_EMA_100","ret_EMA_200","pos_ma_ret", "pos_ma_pct","PCA_ScaledIndex_ma50","PCA_Index_full","fear_greed",
        ]
res = greedy_threshold_search(
    df, feats,
    n_quantiles=24, 
    max_features=6, 
    improvement_eps=0.02,
    adaptive_tails=True, 
    min_count_per_side=200
)
print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])
print("trials_per_feature:", res.logs["trials_per_feature"])


### - By Branches (ex-ML probability): A, B and C

In [ ]:
feats_a = [
    "ATR",'Beta_30', 'Beta_60', 'Beta_90',"Daily_CMF","RSI","St_dev_20","St_dev_5","St_dev_60", "SPY_atr","SPY_rsi","SPY_st_dev","SPY_Agg_Gamma_z_score","SPY_Spot_Gamma_z_score", "dist_Last_px_Arimax_pred_1","dist_Last_px_Arimax_pred_2", "PCA_ScaledIndex_ma50","PCA_Index_full","fear_greed"
    ]
        
feats_b = [ "dist_Last_px_EMA_100", "dist_Last_px_EMA_20","dist_Last_px_EMA_200","dist_Last_px_EMA_50","dist_Last_px_EMA_8", "dist_Last_px_Prev_close_10","dist_Last_px_Prev_close_2","dist_Last_px_Prev_close_3", "dist_Last_px_Prev_close_4","dist_Last_px_Prev_close_5",
    "dist_Last_px_Prev_close_6", "dist_Last_px_Prev_close_7","dist_Last_px_Prev_close_8","dist_Last_px_Prev_close_9", "dist_Last_px_Prev_vwap","dist_Last_px_Var_pred_1","dist_Last_px_Var_pred_2", "dist_Last_px_prev_close","dist_Last_px_prev_high","dist_Last_px_prev_low", 
    "dist_Last_px_prev_open","dist_SPY_px_SPY_close","dist_SPY_px_SPY_ema_20","dist_SPY_px_SPY_ema_200","dist_SPY_px_SPY_ema_50","dist_SPY_px_SPY_ema_8",  "dist_SPY_px_SPY_prev_close"
    ]
           
feats_c = [
    "Premkt_vol_rat","prev_askvol_rat","prev_vol_rat", "Acc_vol_rat","week_day_sin","week_day_cos","ret_Prev_close_2","ret_Prev_close_3", "ret_Prev_close_4","ret_Prev_close_5","ret_Prev_close_6","ret_Prev_close_7", "ret_Prev_close_8","ret_Prev_close_9","ret_Prev_close_10",
    "pos_ret","pos_pct", "ret_EMA_8","ret_EMA_20","ret_EMA_50","ret_EMA_100","ret_EMA_200","pos_ma_ret", "pos_ma_pct", 'momo_px_2', 'momo_px_9',
    ]
        

In [ ]:
res = greedy_threshold_search(df, feats_a, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


In [ ]:
res = greedy_threshold_search(df, feats_b, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


In [ ]:
res = greedy_threshold_search(df, feats_c, n_quantiles=24, max_features=6, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


In [ ]:
feats_a_ml = [ "yhat_train_1", "ATR",'Beta_30', 'Beta_60', 'Beta_90',"Daily_CMF","RSI","St_dev_20","St_dev_5","St_dev_60", "SPY_atr","SPY_rsi","SPY_st_dev","SPY_Agg_Gamma_z_score","SPY_Spot_Gamma_z_score", 
               "dist_Last_px_Arimax_pred_1","dist_Last_px_Arimax_pred_2", "PCA_ScaledIndex_ma50","PCA_Index_full","fear_greed"
    ]
res = greedy_threshold_search(df, feats_a_ml, n_quantiles=24, max_features=3, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


In [ ]:
feats_b_ml = ["yhat_train_1", "dist_Last_px_EMA_100", "dist_Last_px_EMA_20","dist_Last_px_EMA_200","dist_Last_px_EMA_50","dist_Last_px_EMA_8", "dist_Last_px_Prev_close_10","dist_Last_px_Prev_close_2","dist_Last_px_Prev_close_3", "dist_Last_px_Prev_close_4","dist_Last_px_Prev_close_5",
    "dist_Last_px_Prev_close_6", "dist_Last_px_Prev_close_7","dist_Last_px_Prev_close_8","dist_Last_px_Prev_close_9", "dist_Last_px_Prev_vwap","dist_Last_px_Var_pred_1","dist_Last_px_Var_pred_2", "dist_Last_px_prev_close","dist_Last_px_prev_high","dist_Last_px_prev_low", 
    "dist_Last_px_prev_open","dist_SPY_px_SPY_close","dist_SPY_px_SPY_ema_20","dist_SPY_px_SPY_ema_200","dist_SPY_px_SPY_ema_50","dist_SPY_px_SPY_ema_8",  "dist_SPY_px_SPY_prev_close"
    ]

res = greedy_threshold_search(df, feats_b_ml, n_quantiles=24, max_features=3, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


In [ ]:
feats_c_ml = ["yhat_train_1",
    "Premkt_vol_rat","prev_askvol_rat","prev_vol_rat", "Acc_vol_rat","week_day_sin","week_day_cos","ret_Prev_close_2","ret_Prev_close_3", "ret_Prev_close_4","ret_Prev_close_5","ret_Prev_close_6","ret_Prev_close_7", "ret_Prev_close_8","ret_Prev_close_9","ret_Prev_close_10",
    "pos_ret","pos_pct", "ret_EMA_8","ret_EMA_20","ret_EMA_50","ret_EMA_100","ret_EMA_200","pos_ma_ret", "pos_ma_pct",'momo_px_2', 'momo_px_9',
    ]

res = greedy_threshold_search(df, feats_c_ml, n_quantiles=24, max_features=3, improvement_eps=0.02, adaptive_tails=True, min_count_per_side=200)

print("Chosen:", [(c.feat, c.op, round(c.threshold,4)) for c in res.chosen])
print("Sharpe={:.3f}  AnnRet={:.6f}  AnnVol={:.6f}  MaxDD={:.4f}".format(res.sharpe, res.ann_return, res.ann_vol, res.max_dd))
print("crit_lst:", res.crit_list)
print("max_capital_global:", res.logs["max_capital_global"])


### - Bayesian, by Branches: A, B, C (all ex-ML probability) and D (w. ML probability)

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_a, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

for step in res.logs["bo_trace"]:
    print(step)

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_b, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

for step in res.logs["bo_trace"]:
    print(step)

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=feats_c, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])

for step in res.logs["bo_trace"]:
    print(step)

In [ ]:
res = bayesian_greedy_threshold_search(df=df, features=optmz_list_all_plus, mtm_col="mtm_pl", date_col="normed_date", max_features=6, improvement_eps=0.02, seed=123, use_numexpr=True,
                                      # BO knobs
                                      n_calls_per_step=35, n_random_starts=10, tail_penalty=0.4, complexity_penalty=0.03,
                                      # Reproducible locking
                                      lock_to_grid=True, n_quantiles_lock=32,
    )
print("Sharpe:", res.sharpe, "Chosen:", res.chosen)
print("Settings:", res.logs["settings"])
print("crit:", res.crit_list)
print(res.logs["drawdown"])


## 5.2 INS/80% Optimization - Trained sample data

In [ ]:
crit_lst = [
    # {'fear_greed': (63.0, '>='), 'SPY_rsi': (67, '<='), 'St_dev_5': (0.0016, '<='), 'yhat_train_1': (0.40, '>=')}, 
    # {'fear_greed': (60.0, '>='), 'SPY_rsi': (67, '<='), 'St_dev_5': (0.0020, '<='), 'SPY_atr': (1.0, '>='), 'SPY_Agg_Gamma_z_score': (-1.1, '>='), 'dist_Last_px_Arimax_pred_1': (-0.60, '>=')},
    # SELECTED
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>=')},
    
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'momo_px_9': (1, '>=')},
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'momo_ma_9': (1, '>=')},
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'pos_ret': (3, '>=')},
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'pos_ret': (3, '>=')}
    {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'pos_ret': (3, '>=')},
            ]
# Run the optimization loop for the current crit_lst
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_003, crit_lst, 14198000)
# df_test = df_test.sort_values(by=['Sharpe ratio'], ascending=[False], inplace=False)
ins_80.head(20)


In [ ]:
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, 14198000, '80% Optmz', 1, 'short')

## 5.3 INS/20% Optimization - Test sample data

In [ ]:
crit_lst = [
        # {'fear_greed': (60.0, '>='), 'SPY_rsi': (67, '<='), 'St_dev_5': (0.0020, '<='), 'SPY_atr': (1.0, '>='), 'SPY_Agg_Gamma_z_score': (-1.1, '>='), 'dist_Last_px_Arimax_pred_1': (-0.60, '>=')},
        # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>=')},
        # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'pos_ret': (3, '>=')}
        {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'pos_ret': (3, '>=')},
            ]

# Run the optimization loop for the current crit_lst
ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_003, crit_lst, 14198000)
# df_test = df_test.sort_values(by=['Sharpe ratio'], ascending=[False], inplace=False)
ins_20.head(20)


In [ ]:
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, 14198000, '20% Optmz', 1, 'short')

## 5.4 OOS Optimization

In [ ]:
crit_lst = [
    # {'fear_greed': (60.0, '>='), 'SPY_rsi': (67, '<='), 'St_dev_5': (0.0020, '<='), 'SPY_atr': (1.0, '>='), 'SPY_Agg_Gamma_z_score': (-1.1, '>='), 'dist_Last_px_Arimax_pred_1': (-0.60, '>=')},
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='),},
    # {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>=')},
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'pos_ret': (3, '>=')}
    {'fear_greed': (48.0, '>='), 'SPY_rsi': (67, '<='), 'yhat_train_1': (0.61, '>='), 'pos_ret': (3, '>=')},
            ]

# Run the optimization loop for the current crit_lst
oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, 14198000)
# df_test = df_test.sort_values(by=['Sharpe ratio'], ascending=[False], inplace=False)
oos.head(20)


In [ ]:
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, 14198000, 'OOS', 1, 'short')

In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots

datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
# titles = ['G/N Equity Curve: All data', 'G/N Equity Curve: 80%', 'G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20%', 'G/N Equity Curve: 20% Optmz']
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


## 5.5 Alternative optimization parameters

In [ ]:
max_cap = 14198000

crit_lst = [{'yhat_train_1': (0.61, '>='), 'fear_greed': (30.0, '>=')}]


ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_003, crit_lst, max_cap)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, max_cap, '80% Optmz', 1, 'short')

ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_003, crit_lst, max_cap)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, max_cap, '20% Optmz', 1, 'short')

oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, max_cap)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, max_cap, 'OOS', 1, 'short')

In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots

datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
# titles = ['G/N Equity Curve: All data', 'G/N Equity Curve: 80%', 'G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20%', 'G/N Equity Curve: 20% Optmz']
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


In [ ]:
crit_lst = [    
    {'fear_greed': (63.0, '>='), 'SPY_rsi': (67, '<='), 'St_dev_5': (0.0016, '<='), 'yhat_train_1': (0.40, '>=')}
               ]
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_003, crit_lst, max_cap)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, max_cap, '80% Optmz', 1, 'short')

ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_003, crit_lst, max_cap)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, max_cap, '20% Optmz', 1, 'short')

oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, max_cap)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, max_cap, 'OOS', 1, 'short')

In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots

datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
# titles = ['G/N Equity Curve: All data', 'G/N Equity Curve: 80%', 'G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20%', 'G/N Equity Curve: 20% Optmz']
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


In [ ]:
crit_lst = [    
    {'fear_greed': (60.0, '>='), 'SPY_rsi': (64.1621, '<='), 'St_dev_5': (0.0012, '<='), 'SPY_atr': (1.3093, '>='), 'SPY_Agg_Gamma_z_score': (-1.062, '>='), 'dist_Last_px_Arimax_pred_1': (-0.5727, '>=')},
               ]
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_003, crit_lst, max_cap)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, max_cap, '80% Optmz', 1, 'short')

ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_003, crit_lst, max_cap)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, max_cap, '20% Optmz', 1, 'short')

oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, max_cap)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, max_cap, 'OOS', 1, 'short')

In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots

datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
# titles = ['G/N Equity Curve: All data', 'G/N Equity Curve: 80%', 'G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20%', 'G/N Equity Curve: 20% Optmz']
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


In [ ]:

crit_lst = [
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'momo_px_2': (1, '>=')}
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'momo_ma_2': (1, '>=')}
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'momo_px_2': (1, '>='), 'momo_ma_9': (1, '>=')}
    {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'pos_ret': (3, '>=')}
    # {'yhat_train_1': (0.30, '>='), 'fear_greed': (48.0, '>='), 'pos_pct': (0.25, '>=')}
    ]
               
ins_80, optmz_ins_80, optmz_bydate_ins_80 = optmz_loop_wrap(partition_ins_80_003, crit_lst, max_cap)
strat_ins_001, ins_bydate_001 = dashboard(optmz_ins_80, 3.5, max_cap, '80% Optmz', 1, 'short')

ins_20, optmz_ins_20, optmz_bydate_ins_20 = optmz_loop_wrap(partition_ins_20_003, crit_lst, max_cap)
strat_ins_002, ins_bydate_002 = dashboard(optmz_ins_20, 3.5, max_cap, '20% Optmz', 1, 'short')

oos, optmz_oos, optmz_bydate_oos = optmz_loop_wrap(partition_oos_001, crit_lst, max_cap)
strat_oos_001, oos_bydate_001 = dashboard(optmz_oos, 3.5, max_cap, 'OOS', 1, 'short')

In [ ]:
# Grid Equity Curves
fig, axs = plt.subplots(1, 3, figsize=(20, 5))  # Create a 2x3 grid of subplots

datasets = [ins_bydate_001, ins_bydate_002, oos_bydate_001]
# titles = ['G/N Equity Curve: All data', 'G/N Equity Curve: 80%', 'G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20%', 'G/N Equity Curve: 20% Optmz']
titles = ['G/N Equity Curve: 80% Optmz', 'G/N Equity Curve: 20% Optmz', 'G/N Equity Curve: OOS Optmz']

# Plot each dataset on a separate subplot
for i, (data, title) in enumerate(zip(datasets, titles)):
    # ax = axs[i // 3, i % 3]
    ax = axs[i]
    line_chart_grid(ax, data, 'normed_date', 'cum_pl_g', 'cum_pl_n', '$ eqt', title)

# Hide the unused subplot if there is any
if len(datasets) < axs.size:
    for j in range(len(datasets), axs.size):
        fig.delaxes(axs.flatten()[j])
plt.tight_layout()  # Adjust spacing between subplots
plt.show()


In [ ]:
partition_ins_80_003.head()